In [ ]:
# This cell is tagged 'parameters'. Papermill overrides end_date_arg at runtime.
# For manual runs: set end_date_arg = 'YYYY-MM-DD' for a specific date,
# or leave as None to use the default (yesterday with weekday rollback).
end_date_arg = None

In [ ]:
import os
import sys
import pandas as pd

# Add parent directory to path so we can import ports_utils
sys.path.insert(0, os.path.dirname(os.getcwd()))

from ports_utils import make_connection_pool, borrow_connection, compute_dates, PB_DF_SQL as pb_df

start_date, end_date, pre_date = compute_dates(end_date_arg)
print(f'start_date={start_date}, end_date={end_date}, pre_date={pre_date}')

In [ ]:
pool = make_connection_pool(min_conn=1, max_conn=1)

with borrow_connection(pool) as connection:
    PORT_LOANS = pd.read_sql(f'''with forex as (select case
                        when CCY_KEY = 51 then 'EUR'
                        when CCY_KEY = 169 then 'USD' end as CCY,
                    RX
             from ETL_SCHEMA.FX_RATE_FACT
             where SNAP_DATE = DATE '{end_date}'
               and CCY_KEY in (51, 169)),

    studio_clients as (SELECT distinct
        min(R.INT_SNAP_DATE) as INT_SNAP_DATE,
        C.T_VALUE as pcode,
        R.T_HRCODE as studio_hr
    FROM PARTY_SCHEMA.RESPONSIBLE_USER R

             JOIN (select * from ETL_SCHEMA.PARTY_REL_TYPE_REF where src_sys_id='RSL0') T ON R.T_RESPONSIBLE_USER_TYPE_ID = T.SRC_ID

             JOIN PARTY_SCHEMA.PARTY_CODES C ON R.T_PARTYID = C.T_PARTYID AND C.T_TYPE = 'CLIENT_CODE'

    WHERE R.T_ISACTIVE = 1 and  T.LOC_DESCR in ('PREMIUM_ADVISOR','SENIOR_PREMIUM_ADVISOR')
    group by c.T_VALUE, R.T_HRCODE
    ),

           
           
        loans as (select fplf.snap_date,
                          nvl(case
                                  when pos.POS_ID = '101' Then 'Corporate'
                                  when pos.POS_ID = '102' then 'PB'
                                  when sn.id is not null then substr(sn.ID, 1, instr(sn.id, '_') - 1)
                                  else nsit.NEW_SEGMENT end, 'RB')     as segment,
                          nvl(nvl(st.DESCR, nsit.NEW_NAME), bpt.DESCR) as product,
                          fplh.CCY_ID,
                          fplf.POS_ID,
                          fplh.DIR_INTRS_RX_TP_DESCR,
                          fplh.PROD_KEY,
                          (fplf.NOM_AMT * fplf.FX_Rx) as NOM_AMT_GEL,
                          fplf.PERS_ID,
                          fplf.OPEN_DATE,
                          fplf.MAT_DATE,
                          fplf.DIR_INTRS_RX,
                          fplf.PROD_NUM,
                          fplf.CR_OFFC_HR_CODE,
                          st.DESCR                                        new_seg_desc,
                          nsit.NEW_NAME                                   insert_table_name,
                          bpt.DESCR                                       bpT_desct


                   from FINREP_SCHEMA.LOAN_FACT fplf
                            inner join FINREP_SCHEMA.SNAP_DATES s
                                       on s.SNAP_DATE = fplf.SNAP_DATE
                            inner join FINREP_SCHEMA.LOAN_PROD_HIST fplH
                                       on fplH.PROD_KEY = fplf.PROD_KEY
                                           and fplH.SRC_SYS_ID = fplf.SRC_SYS_ID
                                           and fplf.SNAP_DATE between fplh.VALID_FROM and fplh.VALID_TO
                                           and fplh.DEL_FLAG = 0
                            left join REF_SCHEMA.SEGMENT_DIM sT
                                      on st.SEG_KEY = fplh.LMS_PROD_TP_KEY
                                          and st.DEL_FLAG = 0
                            left join REF_SCHEMA.SEGMENT_DIM sN
                                      on sN.SEG_KEY = fplh.LMS_PROD_NAME_KEY
                                          and sn.DEL_FLAG = 0
                            left join REF_SCHEMA.POS_DIM pos
                                      on pos.POS_KEY = fplh.CRNT_POS_KEY
                                          and pos.DEL_FLAG = 0
                            inner join REF_SCHEMA.BUS_PROD_TYPE bpt
                                       on fplh.BUS_PROD_TP_KEY = bpt.BUS_PROD_TP_KEY
                                           and bpt.DEL_FLAG = 0
                            inner join REF_SCHEMA.BUS_PROD_CLASS bpc
                                       on bpc.BUS_PROD_CLASS_KEY = bpt.BUS_PROD_CLASS_KEY
                                           and bpc.DEL_FLAG = 0
                            left join REPORTING_SCHEMA.SEGMENT_OVERRIDE NSIT
                                      on NSIT.PROD_KEY = fplf.PROD_KEY

                   where (PRINC_BAL + OVDU_PRINC_BAL - DIFFERRED_CLN_FEE) > 0
                     and s.snap_date = date '{pre_date}'),


         segment as (select last_day(snap_date) as snap_date,
                            PROD_NUM,
                            dm.SEGMENT
                     from loans l
                              left join REPORTING_SCHEMA.LOAN_SEGMENT_MAP DM
                                        on dm.OLD_SEGMENT = l.segment
                                            and dm.OLD_PRODUCT = l.product),



       REF_LOANS as (
        SELECT * FROM
            (select REF_BY_ID, BUS_KEY, CODE, REF_ID
                        from (select AGRM_ID                as REF_BY_ID,
                                     BUS_PROD_TP_KEY        as BUS_KEY,
                                     PROD_CODE              as CODE,
                                     REFIN_AGREEMENT_NUM_1  as REF_01,
                                     REFIN_AGREEMENT_NUM_2  as REF_02,
                                     REFIN_AGREEMENT_NUM_3  as REF_03,
                                     REFIN_AGREEMENT_NUM_4  as REF_04,
                                     REFIN_AGREEMENT_NUM_5  as REF_05,
                                     REFIN_AGREEMENT_NUM_6  as REF_06,
                                     REFIN_AGREEMENT_NUM_7  as REF_07,
                                     REFIN_AGREEMENT_NUM_8  as REF_08,
                                     REFIN_AGREEMENT_NUM_9  as REF_09,
                                     REFIN_AGREEMENT_NUM_10 as REF_10,
                                     REFIN_AGREEMENT_NUM_11 as REF_11,
                                     REFIN_AGREEMENT_NUM_12 as REF_12,
                                     REFIN_AGREEMENT_NUM_13 as REF_13,
                                     REFIN_AGREEMENT_NUM_14 as REF_14,
                                     REFIN_AGREEMENT_NUM_15 as REF_15
                              from BONUS_SCHEMA.LOAN_DISBURSEMENTS a
                              left join studio_clients  b on a.CLN_PCODE = b.pcode and a.SALES_HR = b.studio_hr
                              where TRN_DATE between date '{start_date}' AND DATE '{end_date}'
                                and REFIN_STAT = 'Y')
                            unpivot (REF_ID for REF_NUM in (
                                REF_01, REF_02, REF_03, REF_04, REF_05,
                                REF_06, REF_07, REF_08, REF_09, REF_10,
                                REF_11, REF_12, REF_13, REF_14, REF_15))) A
                                            LEFT JOIN segment B
    ON A.REF_ID = B.PROD_NUM
                 WHERE SEGMENT = 'RB'
                 ),

         PORT_LOANS_ALL as (
             select distinct PROD_NUM,
                             CR_OFFC_HR_CODE_END as HR_CODE,
                             to_number(POS_ID)   as POS_ID
             from (BONUS_SCHEMA.LOAN_PORTFOLIO)
             where PORT_DATE between DATE '{start_date}' AND DATE '{end_date}'
               and OPEN_DATE >= DATE '{start_date}'
               and OPEN_DATE = PORT_DATE
               and BUS_PROD_TP_KEY in (40, 41, 42, 40099)
               and CR_OFFC_HR_CODE_END not in ('X1001','X1002','X1003','X1004','X1005')
               and SYS_PROD not in (6001, 6002, 6003)
               and ((IE_TP_KEY not in (7001, 7002) or IE_TP_KEY is null)
                 or BUS_PROD_TP_KEY not in (40, 41, 42)
                 or OPEN_DATE >= date '2021-01-01')
               and ((CLT_TP not like '%Real Estate%' and CLT_TP not like '%Vehicle%' and
                     CLT_TP not like '%Deposits%' and open_date between date '2021-01-01' and date '2023-12-31')
                 or CLT_TP is null
                 or BUS_PROD_TP_KEY not in (40, 41, 42))
               and WRITEOFF_FLAG = 0
         ),

         PORT_LOANS_REF as (
             select PROD_NUM,
                    HR_CODE,
                    POS_ID,
                    AMT
             from (
                      select REF_BY_ID,
                             sum(AMT * nvl(FOREX.RX, 1)) as AMT
                      from (
                               (
                                   select REF_BY_ID, REF_ID
                                   from REF_LOANS
                                   where BUS_KEY in (40099, 40, 41, 42)
                                     AND CODE NOT IN (6001, 6002, 6003)
                               ) REF_LOANS
                                   left join (
                                   select PROD_NUM, CCY_ID as CCY, TOT_PRINC_BAL as AMT
                                   from BONUS_SCHEMA.LOAN_PORTFOLIO
                                   where PORT_DATE = DATE '{pre_date}'
                                     and WRITEOFF_FLAG = 0
                                     and pcode not in (select pcode from studio_clients)
                               ) PORT_PREV
                                   on REF_LOANS.REF_ID = PORT_PREV.PROD_NUM
                                   left join FOREX on PORT_PREV.CCY = FOREX.CCY
                               )
                      where PORT_PREV.PROD_NUM is not null
                      group by REF_BY_ID
                  ) PORT_REF
                      left join PORT_LOANS_ALL on PORT_REF.REF_BY_ID = PORT_LOANS_ALL.PROD_NUM
             where PORT_LOANS_ALL.PROD_NUM is not null
               and (HR_CODE not in ({pb_df}) or HR_CODE is null)),

         PORT_LOANS_PREV as (
             select PORT_DATE,
                    PROD_NUM,
                    CR_OFFC_HR_CODE_END                                 as HR_CODE,
                    to_number(POS_ID)                                   as POS_ID,
                    TOT_PRINC_BAL * nvl(FOREX.RX, 1)                    as AMT,
                    case when REFS.REF_ID is not null then 1 else 0 end as IS_REFIN
             from (BONUS_SCHEMA.LOAN_PORTFOLIO) PORT
                      left join FOREX on PORT.CCY_ID = FOREX.CCY
                      left join (select distinct REF_ID from REF_LOANS) REFS on PORT.PROD_NUM = REFS.REF_ID
                  
             where PORT_DATE = DATE '{pre_date}'
               and BUS_PROD_TP_KEY in (40, 41, 42, 40099)
               and SYS_PROD not in (6001, 6002, 6003)
               and ((IE_TP_KEY not in (7001, 7002) or IE_TP_KEY is null)
                 or BUS_PROD_TP_KEY not in (40, 41, 42)
                 or OPEN_DATE >= date '2021-01-01')
               and ((CLT_TP not like '%Real Estate%' and CLT_TP not like '%Vehicle%' and
                     CLT_TP not like '%Deposits%' and open_date between date '2021-01-01' and date '2023-12-31')
                 or CLT_TP is null
                 or BUS_PROD_TP_KEY not in (40, 41, 42))
               and WRITEOFF_FLAG = 0
               and CR_OFFC_HR_CODE_END not in ('X1001','X1002','X1003','X1004','X1005')
               and pcode not in (select pcode from studio_clients)
               and PORT.PROD_NUM not in (select PROD_NUM
    from  BONUS_SCHEMA.LOAN_PORTFOLIO where
    PORT_DATE= DATE '{pre_date}' and (COLL_STAT_DESCR='COLLECTIONS_HARD_LITIGATION'
       or     COLL_STAT_DESCR='COLLECTIONS_HARD' or    COLL_STAT_DESCR='COLLECTIONS_HARD_RESTRUCTURED'))
         ),

         PORT_LOANS_CURR as (
             select PORT_DATE,
                    PROD_NUM,
                    CR_OFFC_HR_CODE_END              as HR_CODE,
                    to_number(POS_ID)                as POS_ID,
                    TOT_PRINC_BAL * nvl(FOREX.RX, 1) as AMT
             from (BONUS_SCHEMA.LOAN_PORTFOLIO) PORT
             left join studio_clients std
             on std.pcode = port.pcode
                      left join FOREX on PORT.CCY_ID = FOREX.CCY
             where PORT_DATE = DATE '{end_date}'
               and BUS_PROD_TP_KEY in (40, 41, 42, 40099)
               and SYS_PROD not in (6001, 6002, 6003)
               and ((IE_TP_KEY not in (7001, 7002) or IE_TP_KEY is null)
                 or BUS_PROD_TP_KEY not in (40, 41, 42)
                 or OPEN_DATE >= date '2021-01-01')
               and ((CLT_TP not like '%Real Estate%' and CLT_TP not like '%Vehicle%' and
                     CLT_TP not like '%Deposits%' and open_date between date '2021-01-01' and date '2023-12-31')
                 or CLT_TP is null
                 or BUS_PROD_TP_KEY not in (40, 41, 42))
               and WRITEOFF_FLAG = 0
           
               and ((TO_CHAR(OPEN_DATE, 'YYYYMM') = TO_CHAR(INT_SNAP_DATE, 'YYYYMM') 
               AND TO_CHAR(OPEN_DATE, 'YYYYMM') = '{month_of_end_date}' and POS_ID not in (103,104)) 
               or std.pcode is null)

               and CR_OFFC_HR_CODE_END not in ('X1001','X1002','X1003','X1004','X1005')
               and (PORT.CR_OFFC_HR_CODE_END not in ({pb_df}) or PORT.CR_OFFC_HR_CODE_END is null)
               and POS_ID <> 'XNA'
            and PORT.PROD_NUM not in (select PROD_NUM
    from  BONUS_SCHEMA.LOAN_PORTFOLIO where
    PORT_DATE= DATE '{pre_date}' and (COLL_STAT_DESCR='COLLECTIONS_HARD_LITIGATION'
       or     COLL_STAT_DESCR='COLLECTIONS_HARD' or    COLL_STAT_DESCR='COLLECTIONS_HARD_RESTRUCTURED'))
         )
     
             select POS_ID,
                    'PORT_LOANS'                as PRODUCT,
                    SUM((CURR - PREV - REF_BY)) as AMT
             from (
                      select PORT_LOANS_REF.PROD_NUM                               as PROD_NUM,
                             nvl(nvl(PORT_LOANS_CURR.HR_CODE, PORT_LOANS_PREV.HR_CODE),
                                 PORT_LOANS_REF.HR_CODE)                           as HR_CODE,
                             nvl(nvl(PORT_LOANS_CURR.POS_ID, PORT_LOANS_PREV.POS_ID),
                                 PORT_LOANS_REF.POS_ID)                            as POS_ID,
                             nvl(PORT_LOANS_CURR.AMT, 0)                           as CURR,
                             nvl(PORT_LOANS_PREV.AMT *
                                 (case
                                      when PORT_LOANS_CURR.PROD_NUM is not null then 1
                                      else (1 - PORT_LOANS_PREV.IS_REFIN) end), 0) as PREV,
                             nvl(PORT_LOANS_PREV.AMT *
                                 (case
                                      when PORT_LOANS_CURR.PROD_NUM is not null then 0
                                      else PORT_LOANS_PREV.IS_REFIN end), 0)       as REF_OF,
                             nvl(PORT_LOANS_REF.AMT, 0)                            as REF_BY
                      from PORT_LOANS_CURR
                               full join PORT_LOANS_PREV on PORT_LOANS_CURR.PROD_NUM = PORT_LOANS_PREV.PROD_NUM
                               full join PORT_LOANS_REF on PORT_LOANS_CURR.PROD_NUM = PORT_LOANS_REF.PROD_NUM)
             where not (CURR = 0 and REF_BY > 0) 
             and (HR_CODE not in ({pb_df}) or HR_CODE is null)
             group by POS_ID
             ''', con=connection)


    # PORT_LOANS.to_excel('PORT_LOANS.xlsx', index=False)
    PORT_LOANS.head()


In [ ]:
os.makedirs('_tmp', exist_ok=True)
output_path = f'_tmp/loans_{end_date}.parquet'
PORT_LOANS.to_parquet(output_path)
print(f'Saved {len(PORT_LOANS)} rows → {output_path}')
